In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
sales = pd.read_excel(
    r"E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\processed\ecommerce_sales_customer_analytics_150k.xlsx",
    sheet_name="processed"
)

df = sales.copy()

In [3]:
print("Rows and columns:", df.shape)

display(df.head())

Rows and columns: (138116, 26)


,order_id,order_date,order_status,sales_channel,customer_id,payment_method,payment_status,currency,shipping_method,warehouse,...,customer_rating,review_sentiment,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,Completed,Mobile App,CUST-003102,Digital Wallet,Paid,USD,Standard,WH-003,...,3.5,Positive,Direct,Default_Campaign,NaN,94,44,12459.68,True,11
1,ORD-773460,2025-12-24,Completed,Website,CUST-003124,Debit Card,Pending,EUR,Economy,WH-005,...,3.6,Positive,Direct,Default_Campaign,NaN,201,171,13032.48,True,11
2,ORD-449374,2021-07-05,Completed,Mobile App,CUST-012496,Cash on Delivery,Paid,USD,Express,WH-015,...,4.3,Positive,YouTube,NaN,NaN,46,16,6159.44,True,6
3,ORD-567636,2023-01-21,Completed,Social Media,CUST-023928,Debit Card,Paid,USD,Express,WH-010,...,3.5,Positive,Email Marketing,NaN,NaN,54,10,5638.30,True,7
4,ORD-820028,2022-05-13,Completed,Social Media,CUST-012730,Digital Wallet,Paid,INR,Standard,WH-013,...,3.0,Neutral,Direct,NaN,NaN,278,147,13240.10,True,8


In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
df.info()

Column names:
['order_id', 'order_date', 'order_status', 'sales_channel', 'customer_id', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_days', 'estimated_delivery_days', 'delivery_status', 'return_status', 'return_reason', 'rating ', 'customer_rating', 'review_sentiment', 'marketing_channel', 'campaign_name', 'coupon_code', 'loyalty_points_earned', 'loyalty_points_redeemed', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 26 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   order_id                 138116 non-null  object        
 1   order_date               138116 non-null  datetime64[ns]
 2   order_status             138116 non-null  object        
 3   sales_channel            138116 non-null  object        
 4   custo

In [5]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

display(
    missing_report.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

print("Exact duplicate rows:", df.duplicated().sum())

,Missing_Count,Missing_Percentage
return_status,128654,93.15
return_reason,128654,93.15
coupon_code,110502,80.01
campaign_name,83233,60.26
review_sentiment,24557,17.78
delivery_days,24557,17.78
estimated_delivery_days,24557,17.78
customer_rating,24557,17.78
is_repeat_customer,0,0.00
customer_lifetime_value,0,0.00


Exact duplicate rows: 0


# Sales Data Cleaning

## 1. Standardize column names

In [6]:
df.columns = df.columns.str.strip()

text_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace(
        r"^\s*$", np.nan, regex=True
    )

print("Column names and text values standardized.")

Column names and text values standardized.


## 2. Clean text values

In [7]:
# Summary statistics for numerical columns
display(df.describe().T)

,count,mean,min,25%,50%,75%,max,std
order_date,138116,2023-07-21 10:11:31.662081280,2021-01-01 00:00:00,2022-04-20 00:00:00,2023-08-01 00:00:00,2024-10-29 00:00:00,2025-12-31 00:00:00,NaN
delivery_days,113559.0,4.557772,0.0,2.0,4.0,7.0,14.0,2.810012
estimated_delivery_days,113559.0,4.298875,0.0,2.0,4.0,6.0,10.0,2.59408
rating,138116.0,3.068486,0.0,3.0,4.0,4.0,5.0,1.514408
customer_rating,113559.0,3.677015,1.6,3.4,3.7,4.0,5.0,0.483672
loyalty_points_earned,138116.0,103.991051,0.0,17.0,70.0,151.0,1621.0,115.17564
loyalty_points_redeemed,138116.0,51.934128,0.0,3.0,25.0,71.0,1063.0,73.217692
customer_lifetime_value,138116.0,8971.639047,22.55,5713.57,8392.525,11651.03,30268.44,4430.506588
customer_order_count,138116.0,6.990559,1.0,5.0,7.0,9.0,20.0,2.444166


In [8]:
# Summary statistics for categorical columns
display(df.describe(include=["object", "string"]).T)

,count,unique,top,freq
order_id,138116,138116,ORD-301242,1
order_status,138116,4,Completed,113559
sales_channel,138116,4,Mobile App,55318
customer_id,138116,24911,CUST-014687,19
payment_method,138116,7,Credit Card,41460
payment_status,138116,4,Paid,104250
currency,138116,7,USD,82600
shipping_method,138116,4,Standard,62006
warehouse,138116,19,WH-014,7447
delivery_status,138116,4,On Time,86916


In [10]:
# Check unique values in important categorical columns
columns = [
    "order_status",
    "sales_channel",
    "payment_method",
    "payment_status",
    "shipping_method",
    "delivery_status",
    "return_status"
]

for col in columns:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


order_status
order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64

sales_channel
sales_channel
Mobile App      55318
Website         48237
Marketplace     20698
Social Media    13863
Name: count, dtype: int64

payment_method
payment_method
Credit Card          41460
Debit Card           27635
PayPal               20750
Digital Wallet       16382
Cash on Delivery     13863
Buy Now Pay Later    11061
Bank Transfer         6965
Name: count, dtype: int64

payment_status
payment_status
Paid        104250
Pending      13975
Failed       10429
Refunded      9462
Name: count, dtype: int64

shipping_method
shipping_method
Standard    62006
Express     34373
Economy     27824
Same Day    13913
Name: count, dtype: int64

delivery_status
delivery_status
On Time      86916
Cancelled    24557
Delayed      16886
Early         9757
Name: count, dtype: int64

return_status
return_status
NaN         128654
Returned      9462
Name: count,

## 3. Data Quality Analysis

In [11]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
})

display(
    missing_report[missing_report["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

,Missing_Count,Missing_Percentage
return_status,128654,93.15
return_reason,128654,93.15
coupon_code,110502,80.01
campaign_name,83233,60.26
delivery_days,24557,17.78
estimated_delivery_days,24557,17.78
customer_rating,24557,17.78
review_sentiment,24557,17.78


In [12]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Total rows:", len(df))
print("Unique order IDs:", df["order_id"].nunique())
print("Unique customers:", df["customer_id"].nunique())


Exact duplicate rows: 0
Total rows: 138116
Unique order IDs: 138116
Unique customers: 24911


In [13]:
df["order_date"] = pd.to_datetime(
    df["order_date"], errors="coerce"
)

print("Missing or invalid dates:", df["order_date"].isna().sum())
print("Earliest order:", df["order_date"].min())
print("Latest order:", df["order_date"].max())

Missing or invalid dates: 0
Earliest order: 2021-01-01 00:00:00
Latest order: 2025-12-31 00:00:00


In [15]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Check for duplicate column names
print("Duplicate column names:", df.columns.duplicated().sum())

print(df.columns.tolist())


Duplicate column names: 0
['order_id', 'order_date', 'order_status', 'sales_channel', 'customer_id', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_days', 'estimated_delivery_days', 'delivery_status', 'return_status', 'return_reason', 'rating', 'customer_rating', 'review_sentiment', 'marketing_channel', 'campaign_name', 'coupon_code', 'loyalty_points_earned', 'loyalty_points_redeemed', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']


In [16]:
text_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    df[col] = df[col].str.strip()

    df[col] = df[col].replace(
        r"^\s*$", np.nan, regex=True
    )

print("Text formatting completed.")

Text formatting completed.


In [17]:
# Convert numerical columns
numeric_cols = [
    "delivery_days",
    "estimated_delivery_days",
    "customer_rating",
    "customer_lifetime_value",
    "customer_order_count"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

print("Data type conversion completed.")
df.info()

Data type conversion completed.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 26 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   order_id                 138116 non-null  object        
 1   order_date               138116 non-null  datetime64[ns]
 2   order_status             138116 non-null  object        
 3   sales_channel            138116 non-null  object        
 4   customer_id              138116 non-null  object        
 5   payment_method           138116 non-null  object        
 6   payment_status           138116 non-null  object        
 7   currency                 138116 non-null  object        
 8   shipping_method          138116 non-null  object        
 9   warehouse                138116 non-null  object        
 10  delivery_days            113559 non-null  float64       
 11  estimated_delivery_days  113559 non-null  floa

In [18]:
display(
    df[[
        "order_status",
        "return_status",
        "return_reason"
    ]].head(20)
)

print("\nReturn status:")
print(df["return_status"].value_counts(dropna=False))

print("\nOrder status:")
print(df["order_status"].value_counts(dropna=False))

,order_status,return_status,return_reason
0,Completed,NaN,NaN
1,Completed,NaN,NaN
2,Completed,NaN,NaN
3,Completed,NaN,NaN
4,Completed,NaN,NaN
5,Pending,NaN,NaN
6,Completed,NaN,NaN
7,Completed,NaN,NaN
8,Completed,NaN,NaN
9,Completed,NaN,NaN



Return status:
return_status
NaN         128654
Returned      9462
Name: count, dtype: int64

Order status:
order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64


In [20]:
for col in [
    "coupon_code",
    "campaign_name",
    "review_sentiment",
    "customer_rating",
    "delivery_days"
]:
    if col in df.columns:
        print(f"\n{col}:")
        print("Missing:", df[col].isna().sum())
        print("Available:", df[col].notna().sum())


coupon_code:
Missing: 110502
Available: 27614

campaign_name:
Missing: 83233
Available: 54883

review_sentiment:
Missing: 24557
Available: 113559

customer_rating:
Missing: 24557
Available: 113559

delivery_days:
Missing: 24557
Available: 113559


In [23]:
df.to_csv(
    "cleaned_sales.csv",
    index=False
)

print("Saved cleaned_sales.csv")

Saved cleaned_sales.csv
